# Занятие 13. Выделение движения и оптический поток

За 90 минут: **маска MOG2/KNN → соответствия LK → Farnebäck/EPE → клиника HW4**.
Три TODO по 12/11/7 строк; свой итерационный LK — ⭐ по желанию.
Данные генерируются в памяти, камера, видеофайл и сеть не нужны.
Выберите ядро `iu-intro-cv/.venv`; зависимости заранее — из `docs/setup-guide.md`.

Опора: L13, **страницы публичного PDF** 8–19 (фон), 24–40 (LK), 45–48 (плотный поток/EPE).
В PPTX число слайдов отличается; номера ниже относятся к PDF.
Демо 01–08 дают API; числа этого нового набора измеряются отдельно.
План: 15 минут опорного примера, 50 TODO, 15 разбора (последние 5 — вопросы HW3),
10 старта клиники HW4. Формат при занятии с quiz — в методичке TA.

In [ ]:
import importlib.util
import importlib.metadata
import json
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

assert cv2.__version__ == '4.14.0', 'Выберите ядро iu-intro-cv/.venv'
assert importlib.metadata.version('opencv-contrib-python') == '4.14.0.94'
cv2.setNumThreads(1)
if 'get_ipython' in globals():
    get_ipython().run_line_magic('matplotlib', 'inline')
if 'LAB_DIR' not in globals():
    LAB_DIR = next(p for p in (Path.cwd(), Path.cwd() / 'labs/lab13-motion')
                   if (p / 'make.py').is_file())
spec = importlib.util.spec_from_file_location('lab13_data', LAB_DIR / 'make.py')
data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)
make_scene, array_hashes = data_module.make_scene, data_module.array_hashes

In [ ]:
def mask_report(prediction, truth):
    prediction, truth = np.asarray(prediction, bool), np.asarray(truth, bool)
    if prediction.shape != truth.shape:
        raise ValueError('Mask shapes differ')
    tp = int(np.count_nonzero(prediction & truth))
    fp = int(np.count_nonzero(prediction & ~truth))
    fn = int(np.count_nonzero(~prediction & truth))
    union = tp + fp + fn
    return dict(tp=tp, fp=fp, fn=fn, iou=tp / union if union else 1.)


def flow_report(prediction, truth, mask):
    prediction, truth, mask = np.asarray(prediction), np.asarray(truth), np.asarray(mask, bool)
    if prediction.shape != truth.shape or truth.shape != (*mask.shape, 2):
        raise ValueError('Expected flow (...,2) and matching GT mask')
    if not mask.any() or not np.isfinite(truth[mask]).all():
        raise ValueError('Empty or invalid GT support')
    finite = np.isfinite(prediction).all(axis=-1)
    evaluated = mask & finite
    errors = np.linalg.norm(prediction[evaluated] - truth[evaluated], axis=-1)
    coverage = float(evaluated.sum() / mask.sum())
    conditional = float(errors.mean()) if len(errors) else None
    return dict(epe=conditional if coverage == 1. else None, conditional_epe=conditional,
                coverage=coverage, gt_count=int(mask.sum()))


def in_frame(points, shape):
    points = np.asarray(points).reshape(-1, 2)
    return (np.isfinite(points).all(axis=1) & (points[:, 0] >= 0) & (points[:, 0] < shape[1])
            & (points[:, 1] >= 0) & (points[:, 1] < shape[0]))


def fb_check(a, b, points, q, status, level, limit):
    valid = (status.ravel() == 1) & in_frame(points, a.shape) & in_frame(q, b.shape)
    ids = np.flatnonzero(valid)
    distance = np.full(len(points), np.inf)
    if len(ids):
        back, back_status, _ = cv2.calcOpticalFlowPyrLK(
            b, a, q[ids].reshape(-1, 1, 2), None, winSize=(21, 21), maxLevel=level,
            criteria=(cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 40, .001))
        good = (back_status.ravel() == 1) & in_frame(back, a.shape)
        distance[ids[good]] = np.linalg.norm(back[good].reshape(-1, 2) - points[ids[good]], axis=1)
    return valid & (distance < limit), distance


def track_report(result, shift, eligible=None):
    eligible = np.ones(len(result['points']), bool) if eligible is None else np.asarray(eligible, bool)
    if eligible.shape != result['valid'].shape or not eligible.any():
        raise ValueError('No eligible tracks')
    valid = eligible & result['valid']
    error = np.linalg.norm(result['q'][valid] - result['points'][valid] - shift, axis=1)
    return dict(eligible=int(eligible.sum()), kept=int(valid.sum()),
                coverage=float(valid.sum() / eligible.sum()),
                conditional_epe=float(error.mean()) if len(error) else None)


def flow_color(flow, scale=10.):
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError('Scale must be finite and positive')
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*flow.shape[:2], 3), np.uint8)
    hsv[..., 0] = np.uint8(np.mod(angle * 90 / np.pi, 180))
    hsv[..., 1] = 255
    hsv[..., 2] = np.uint8(np.clip(magnitude * 255 / scale, 0, 255))
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

## 0–15. Что считать движением?

Фон (background) — история наблюдений при фиксированной камере. Кадровая разность
видит изменившиеся пиксели: движущийся объект оставляет две полосы, остановившийся исчезает.
В ролике сначала 24 кадра только фона, затем прямоугольник с тенью, с кадра 46 он стоит.
GT — полный силуэт объекта; тень считается фоном. Импульсы имитируют битые пиксели.
Модель получает только изображения; GT используется после предсказания.

In [ ]:
scene = make_scene()
assert scene['video'].shape == (80, 96, 144, 3)
frame_index = 38
gray = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in scene['video']])
moving_difference = cv2.absdiff(gray[frame_index - 1], gray[frame_index]) > 25
stopped_difference = cv2.absdiff(gray[-2], gray[-1]) > 25
diff_reports = {'moving': mask_report(moving_difference, scene['video_gt'][frame_index]),
                'stopped': mask_report(stopped_difference, scene['video_gt'][-1])}
print(json.dumps(diff_reports, indent=2))
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, image, title in zip(axes, [scene['video'][frame_index][..., ::-1], moving_difference, stopped_difference],
                            ['Moving object + shadow', 'Frame difference: moving', 'Frame difference: stopped']):
    ax.imshow(image, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## TODO 1 · 15–32. MOG2/KNN и маски

Создайте модель с `history` и порогом, прогрейте на первых 24 кадрах.
Для MOG2 порог — `varThreshold=16`, для KNN — `dist2Threshold=400`: это разные величины.
`apply` возвращает 0 (фон), 127 (тень) и 255 (объект). Выбирайте **только 255**,
затем открытие (opening) ядром 3×3; верните сырые и очищенные маски всех кадров.
Нельзя обучить модель сразу на всех кадрах и затем оценивать прошлое.
IoU=TP/(TP+FP+FN); сравните FP и FN до/после, морфология может удалить тонкие детали.

In [ ]:
def background_masks(frames, kind='MOG2', history=60, threshold=None, adapt=.01, warmup=24):
    frames = np.asarray(frames)
    if frames.ndim != 4 or frames.shape[-1] != 3 or frames.dtype != np.uint8 or not len(frames):
        raise ValueError('Expected nonempty uint8 BGR video (T,H,W,3)')
    if kind not in ('MOG2', 'KNN') or history < 1 or not 0 <= adapt <= 1 or warmup < 1:
        raise ValueError('Invalid background configuration')
    threshold = (16. if kind == 'MOG2' else 400.) if threshold is None else float(threshold)
    if threshold <= 0:
        raise ValueError('Threshold must be positive')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 12 строк: модель, прогрев, только label=255, морфология
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────

In [ ]:
bg = {kind: background_masks(scene['video'], kind) for kind in ('MOG2', 'KNN')}
bg_reports = {}
for kind, result in bg.items():
    report = mask_report(result['clean'][frame_index] == 255, scene['video_gt'][frame_index])
    report['raw_fp'] = mask_report(result['raw'][frame_index] == 255, scene['video_gt'][frame_index])['fp']
    report['shadow_labels'] = int(np.count_nonzero(result['raw'][frame_index] == 127))
    bg_reports[kind] = report
    assert result['raw'].shape == scene['video_gt'].shape
    assert report['iou'] > .75 and report['fp'] < 150
print(json.dumps(bg_reports, indent=2))
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for row, kind in enumerate(bg):
    for col, (image, title) in enumerate([(scene['video_gt'][frame_index], 'GT'),
          (bg[kind]['raw'][frame_index], 'labels 0/127/255'), (bg[kind]['clean'][frame_index], 'opening')]):
        axes[row,col].imshow(image, cmap='gray', vmin=0, vmax=255 if col else 1)
        axes[row,col].set_title(kind + ': ' + title); axes[row,col].axis('off')
plt.tight_layout(); plt.show()
# A stopped object can be absorbed by adaptation; this is a controlled comparison.
adaptation = {}
for rate in (0., .1):
    output = background_masks(scene['video'], adapt=rate)
    adaptation[str(rate)] = mask_report(output['clean'][-1] == 255, scene['video_gt'][-1])
print('Stopped object, frozen / adapting model:', adaptation)
background_grid = []
for history in (30, 90):
    for threshold in (8., 24.):
        output = background_masks(scene['video'],history=history,threshold=threshold)
        row = mask_report(output['clean'][frame_index] == 255,scene['video_gt'][frame_index])
        background_grid.append(dict(history=history,varThreshold=threshold,**row))
print('MOG2 grid (explicit learningRate controls adaptation):',background_grid)

## TODO 2 · 32–50. Разреженный пирамидальный Лукас–Канаде

`goodFeaturesToTrack` ищет углы. Для них M из L8 имеет два достаточно больших
собственных числа. Вход LK — `float32`, `(N,1,2)`; выход — точки второго кадра.
`q-p=(u,v)`, единицы px/frame, положительные направления вправо/вниз.
`maxLevel=0` отключает пирамиду. `status=1` сообщает об успехе внутренней проверки,
но не гарантирует правильное соответствие. `err` по умолчанию — фотометрическая
ошибка патча, **не EPE**, при `status=0` её не интерпретируют.
Готовая `fb_check` проверяет границы и обратный проход; порог FB=1 px.
Она не видит GT. Для среднего EPE принятых точек всегда сообщайте покрытие.

In [ ]:
def sparse_lk(a, b, points=None, max_level=3, fb_limit=1.):
    if a.shape != b.shape or a.ndim != 2 or a.dtype != np.uint8 or b.dtype != np.uint8:
        raise ValueError('Expected equal grayscale uint8 images')
    if max_level < 0 or fb_limit <= 0:
        raise ValueError('Invalid LK parameters')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 11 строк: точки, пирамидальный LK, status и обратная проверка
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────

In [ ]:
points = cv2.goodFeaturesToTrack(scene['a'], 80, .02, 10, mask=np.uint8(scene['valid']) * 255).reshape(-1, 2)
tracked = sparse_lk(scene['a'], scene['b'], points)
lk_report = track_report(tracked, scene['shift'])
assert lk_report['coverage'] > .9 and lk_report['conditional_epe'] < .15
occluded_tracks = sparse_lk(scene['a'], scene['occluded'], points)
point_pixels = np.rint(points).astype(int)
visible_points = scene['visible'][point_pixels[:,1], point_pixels[:,0]]
occlusion_report = track_report(occluded_tracks, scene['shift'], visible_points)
print('Clean LK:', lk_report)
print('Occluded: status accepted', int(occluded_tracks['status'].sum()), 'FB kept', int(occluded_tracks['valid'].sum()))
print('GT-visible points (evaluation only):', occlusion_report)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, image, result, title in [(axes[0],scene['b'],tracked,'Clean LK'),
                                (axes[1],scene['occluded'],occluded_tracks,'Occlusion + FB')]:
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    ids = result['valid']; p, q = result['points'][ids], result['q'][ids]
    ax.quiver(p[:,0], p[:,1], q[:,0]-p[:,0], q[:,1]-p[:,1], angles='xy', scale_units='xy', scale=1, color='lime')
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
large_scene = make_scene(shift=(18., -9.))
large_points = cv2.goodFeaturesToTrack(large_scene['a'],80,.02,10,mask=np.uint8(large_scene['valid'])*255)
pyramid_reports = {str(level): track_report(sparse_lk(large_scene['a'],large_scene['b'],large_points,max_level=level),
                                          large_scene['shift']) for level in (0, 3)}
print('Large displacement, compare EPE AND coverage:', pyramid_reports)

## TODO 3 · 50–65. Плотный поток Фарнебека (Farnebäck)

Заполните 7 строк: поток `(H,W,2)`, полярные координаты и HSV-визуализация.
Для OpenCV H лежит в 0…179, направление в радианах умножается на 90/π.
V — модуль с фиксированной шкалой 10 px/frame; S=255. Для matplotlib верните RGB.
Отдельно готовая `flow_report` считает `EPE=mean(norm(flow-GT))` на фиксированной
GT-маске совместной видимости с запасом для окон. Алгоритм маску не получает.
Не выкидывайте плохие предсказания из знаменателя: при NaN полный EPE=null,
условный EPE и coverage сообщаются отдельно. Для нулевого потока EPE=norm(GT).

In [ ]:
def dense_pair(a, b, scale=10.):
    if a.shape != b.shape or a.ndim != 2 or a.dtype != np.uint8 or b.dtype != np.uint8:
        raise ValueError('Expected equal grayscale uint8 images')
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError('Scale must be finite and positive')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 7 строк: плотный поток и HSV с общей шкалой px/frame
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────

In [ ]:
dense = dense_pair(scene['a'], scene['b'])
dense_report = flow_report(dense['flow'], scene['gt'], scene['valid'])
zero_report = flow_report(np.zeros_like(scene['gt']), scene['gt'], scene['valid'])
assert dense_report['coverage'] == 1. and dense_report['epe'] < .4
assert abs(zero_report['epe'] - np.linalg.norm(scene['shift'])) < 1e-5
print('Farneback:', dense_report, 'zero baseline:', zero_report)
error_map = np.linalg.norm(dense['flow'] - scene['gt'], axis=-1)
error_map[~scene['valid']] = np.nan
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].imshow(dense['rgb']); axes[0].set_title('Flow: fixed 10 px/frame')
im = axes[1].imshow(error_map, vmin=0, vmax=.4); axes[1].set_title('EPE on fixed GT support'); fig.colorbar(im, ax=axes[1])
axes[2].imshow(scene['valid'], cmap='gray'); axes[2].set_title('GT support (source domain)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 65–75. Апертурная проблема и границы применимости

Готовый пример ниже: вертикальный край движется вправо/вниз, но движение вдоль
него ничего не меняет. В окне есть только `Ix`, а `Iy=0`: M вырождена.
Расширить окно до угла — получить второе независимое направление градиента.
Пирамида помогает при больших смещениях, пока детали различимы на грубом уровне.
FB может пропустить неверную пару при повторяющейся текстуре; это проверка согласованности,
а не доказательство истинности. На реальном видео движение камеры меняет весь фон;
блики/тени и окклюзия нарушают предпосылки яркостного постоянства.

In [ ]:
edge = scene['edge']
shifted_x = data_module.translate(edge, (3, 0))
shifted_xy = data_module.translate(edge, (3, 5))
assert np.array_equal(shifted_x[12:-12,12:-12], shifted_xy[12:-12,12:-12])
gx = cv2.Sobel(edge.astype(np.float32),cv2.CV_32F,1,0,ksize=3,scale=1/8)
gy = cv2.Sobel(edge.astype(np.float32),cv2.CV_32F,0,1,ksize=3,scale=1/8)
A = np.column_stack([gx[20:44,34:46].ravel(), gy[20:44,34:46].ravel()])
print('Edge M eigenvalues:', np.linalg.eigvalsh(A.T @ A))
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, image, title in zip(axes, [edge, shifted_x, shifted_xy], ['Edge','shift (3,0)','shift (3,5)']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 80–90. Клиника HW4: фиксированная маска и единицы

Это старт разбора HW4 на занятии 13; основной слот — занятие 14 по актуальному TA-плану.
Оптический поток измеряет **перенос между моментами времени**, стерео-диспаратность —
**между камерами**. Их карты нельзя подменять друг другом.

| Что проверили здесь / ранее | Функция шаблона HW4 |
|---|---|
| Общая GT-маска, пропуски, отчёт покрытия | `disparity_metrics` — bad-2.0 и RMSE, не EPE |
| Lab11: projectPoints, per-view RMS и повторная калибровка | `reprojection_errors`, `calibrate` |
| Lab11: координаты доски и лучи камеры | `plane_measure` |
| Lab12: int16 /16, SGBM/WLS, метры базы | `sgbm`, `stereo_depth`, `point_cloud` |

Открывайте `homeworks/hw4-calib3d/sample_data/README.md` и `case1/expected.json`.
Базовые пары уже ректифицированы; GT из той же пары, без повторного варпа.
Ниже — два учебных контроля с известным ответом, без эталона ДЗ:
1) искусственные ошибки `(3,4)` дают RMS **5 px**, а не RMSE по отдельным координатам;
2) ошибки диспаратности ровно 2 px ещё не bad-2.0, NaN считается пропуском/ошибкой;
3) при `f=400 px`, `B=.12 m`, `d=24 px` глубина 2 m; это заданный расчётный пример.
Перенесите правила в свой отчёт, сохранив единицы и маску; выбросы калибровки
объясняйте по конкретным кадрам, сравнивайте RMS и K до/после.
75–80 минут отведены вопросам HW3 по действующему плану: откройте публичные README
и sample_data HW3, разберите вопрос о нормировке H или шумной репроекции.
Полный эталон ДЗ для этого не требуется; блок ниже начинает разбор HW4.

In [ ]:
def clinic_disparity(prediction, truth, valid):
    prediction, truth, valid = np.asarray(prediction), np.asarray(truth), np.asarray(valid, bool)
    if prediction.shape != truth.shape or truth.shape != valid.shape or not valid.any():
        raise ValueError('Invalid disparity shapes/support')
    valid = valid & np.isfinite(truth) & (truth > 0)
    if not valid.any():
        raise ValueError('No valid GT')
    finite = np.isfinite(prediction) & (prediction > 0)
    overlap = valid & finite
    errors = np.abs(prediction[overlap] - truth[overlap])
    missing = int(np.count_nonzero(valid & ~finite))
    conditional = float(np.sqrt(np.mean(errors**2))) if len(errors) else None
    return dict(bad2=100. * (np.count_nonzero(errors > 2.) + missing) / valid.sum(),
                coverage=100. * overlap.sum() / valid.sum(),
                rmse=conditional if missing == 0 else None, conditional_rmse=conditional)

In [ ]:
residual = np.array([[3.,4.], [3.,4.]])
reprojection_rms = float(np.sqrt(np.mean(np.sum(residual**2, axis=1))))
clinic_report = clinic_disparity(np.array([8.,10.,11.,np.nan]), np.full(4,8.), np.ones(4,bool))
depth_m = 400. * .12 / 24.
assert reprojection_rms == 5. and depth_m == 2.
assert clinic_report['bad2'] == 50. and clinic_report['coverage'] == 75. and clinic_report['rmse'] is None
print('RMS px / disparity / depth m:', reprojection_rms, clinic_report, depth_m)

In [ ]:
# Public HW4 data only; the lab itself remains usable if copied on its own.
hw4_candidates = [LAB_DIR.parents[2] / 'iu-intro-cv/homeworks/hw4-calib3d/sample_data/case1',
                  LAB_DIR.parents[1] / 'homeworks/hw4-calib3d/sample_data/case1']
case_dir = next((p for p in hw4_candidates if (p/'expected.json').is_file()),None)
if case_dir is not None:
    expected_case = json.loads((case_dir/'expected.json').read_text(encoding='utf-8'))
    per_view = np.array(expected_case['calibration']['per_image_before_px'])
    print('HW4 case1, public expected per-view RMS:',per_view)
    print('Inspect the worst view before rejecting:',int(np.argmax(per_view)),float(per_view.max()))
    with np.load(case_dir/'gt.npz',allow_pickle=False) as gt_file:
        print('HW4 case1 GT arrays (keep units):',{key:list(gt_file[key].shape) for key in gt_file.files})
    print('Compare your output with case1/expected.json; numbers from lab13 are not HW4 targets.')
else:
    print('Open HW4 sample_data/case1 from the course repository for the clinic.')

## ⭐ Свой пирамидальный LK для одной точки

По желанию напишите `lk_point(a,b,point,levels=2,radius=5,iterations=30)`:
на каждом уровне извлеките патч и градиенты, соберите `A=[Ix,Iy]`, `M=A.T@A`,
решите `M delta=A.T@(template-warped)` и уточните `d+=delta`.
При переходе к следующему масштабу умножьте d на 2; проверьте собственные числа,
границы и остановку по norm(delta). Извлечение патча и Sobel/pyrDown можно взять из OpenCV;
готовые функции оптического потока для собственной реализации не использовать.
Проверьте переносы `(0.4,-0.3)`, `(6,3)` и `(12,-6)`, новый seed и плоский/прямой край.
Ориентир для данной синтетики: EPE <0.4 px при хорошем патче, вырожденный M → явный отказ.

Перед показом результата: объясните FP/FN маски, status/err/FB, знак `(u,v)`,
фиксированную область EPE и различие EPE от bad-2.0/RMSE в HW4.
Литература: L13 PDF и восемь локальных демо; Lucas–Kanade (1981), Farnebäck (2003).